# 17.11 Activation 也量化會改變什麼？

# 第 17 章：量化，用更少位元表示模型

前置：第2章Linear與數值。像把連續刻度尺改成較粗的格子：先看誤差，再看真正的儲存bytes，最後才談硬體加速。quantization.py的參考Linear會反量化，不會假裝是低位元kernel。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：換較粗刻度尺：省格子但有誤差**

縮小儲存與硬體加速分開驗證。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/quantization.svg")))

**先想一想：**只在一種短輸入量到的range能代表所有輸入嗎？

activation每次輸入不同，範圍也變化。權重與activation誤差會一起進入輸出，不宜只用一張weight誤差圖判斷模型表現。

**把直覺寫成數學：**y_hat=Q(x)·Q(W)ᵀ；多個層還會累積誤差。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.quantization import quantize_symmetric

x = torch.randn(2, 4)
w = torch.randn(3, 4)
qx, sx = quantize_symmetric(x, 4)
qw, sw = quantize_symmetric(w, 4)
print("weight-only誤差", (x @ w.T - x @ (qw * sw).T).abs().mean().item())
print("weight+activation誤差", (x @ w.T - (qx * sx) @ (qw * sw).T).abs().mean().item())

**如何讀結果：**不同誤差方向可能抵銷，平均誤差不必每例單調增加。

**只改一件事：**只放大一筆activation。
